In [ ]:
# === COMPLETE END-TO-END EXAMPLE ===
from utils.chunk_processor import ChunkProcessor, ABConfig

processor = ChunkProcessor(
    ab_config=ABConfig(
        conversion_event='purchase',
        user_id_col='user_id',
        variant_assignment=lambda uid: np.where(uid % 2 == 0, 'A', 'B')
    )
)

# Ingestion (run once)
chunks = processor.chunk_csv_to_parquet('data/2019-Nov.csv')
processor.load_chunks_to_duckdb(chunks, db_path='data/marketplace-analytics.duckdb')

# Run dbt (shell or subprocess)
# dbt run

# Analysis (same as demo path)
results = processor.run_ab_aggregation(db_path='data/marketplace-analytics.duckdb')


# 4. Results (production-ready dict)
print(f"Global A/B Test Results:")
print(f"CR A: {results['cr_A']:.3%} ({results['total_conv_A']:,}/{results['total_users_A']:,})")
print(f"CR B: {results['cr_B']:.3%} ({results['total_conv_B']:,}/{results['total_users_B']:,})")
print(f"Relative uplift: {results['relative_uplift_pct']:.1f}%")
print(f"p-value: {results['p_value']:.4f} ({'significant' if results['alpha_005_significant'] else 'not significant'} at α=0.05)")

# Done! results dict ready for charts, dashboards, etc.